# Моделювання газу твердих сфер

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from random import random

Константи

In [22]:
k = 1.38e-23

Вхідні дані

In [23]:
S = np.pi
H = 1
N = 1000
R = 1e-6
M = 1e-6
T = 273
G = 9.8

Змінні та функції для зручності

In [29]:
# Діаметр основи циліднра
D = 2 * np.sqrt(S / np.pi)

# Випадковий одиничний вектор
def random_k():
    k = np.array([random() - 1, random() - 1, random() - 1])
    return k / np.linalg.norm(k)

In [41]:
# Кінетична енергія від швидкості
def K(v):
    return M*v**2/2

# Швидкість від кінетичної енергії
def v(K):
    return np.sqrt(2*K/M)

In [43]:
# Закон збереження імпульсу
def update_velocities(p1, p2, v1, v2):
    p = (p1 - p2) / np.linalg.norm(p1 - p2)
    d = np.array([[0,-1], [1, 0]]) @ p
    v1new = v1 @ d * d + (v1 + v2) @ p * p / 2 
    v2new = v1 @ d * d - (v1 + v2) @ p * p / 2 
    return v1new, v2new

Генерування положення кульок (в межах посудини) і їхні швидкості випадковим чином, але так, 
щоб сума їхніх кінетичних енергії була рівна $3NkT/2$ (N--кількість частинок, T--температура, k--стала Больцмана)

In [34]:
max_enegry = 3 * N * k * T / 2
energies = np.array([random() * max_enegry for _ in range(N)])
energies /= sum(energies)

In [35]:
positions = [np.array([(random() - 1) * D, (random() - 1) * D, (random() - 1) * H]) for _ in range(N)]
velocities = [random_k() * v(K) for K in energies]

Оновлення позицій 

In [36]:
def new_positions(positions, velocities):
    dt = R / 4 / max(velocities)
    return positions + velocities * dt

Перевірка на зіткнення

In [39]:
def detect_collisions(positions):
    collisions_matrix = np.zeros((N, N))
    for i, p1 in enumerate(positions[:-2]):
        for j, p2 in enumerate(positions[i+1:]):
            if np.linalg.norm(positions[i] - positions[i + j]) <= 2*R:
                collisions_matrix[i][i + j] = 1
                collisions_matrix[i + j][i] = 1

Оновлення швидкостей

In [45]:
from copy import deepcopy

def new_velocities(positions, collisions_matrix, velocities):
    new_velocities = deepcopy(velocities)
    for i in range(N):
        for j in range(N): 
            if collisions_matrix[i, j]:
                velocities[i, j], velocities[j, i] = update_velocities(
                    positions[i],
                    positions[j],
                    velocities[i],
                    velocities[j])

### имуляція

Параметри симуляції

In [ ]:
simulation_time = 10

### а) графік розподілу частинок по швидкостях (розподіл Максвела)